In [2]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import torch.optim as optim
import segmentation_models_pytorch as smp
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

c:\Users\40757\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def RLE_to_mask(rle,shape=(256, 1600)):
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths

    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order='F')

In [4]:
class SteelDataset(Dataset):
    def __init__(self, df, img_folder, transform=None):
        self.df = df
        self.img_folder = img_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.img_folder}/{row['ImageId']}"
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # 1. Create the mask
        mask = RLE_to_mask(row['EncodedPixels'], shape=(256, 1600))
        
        # 2. Extract Bounding Box from Mask
        pos = np.where(mask)
        xmin, xmax = np.min(pos[1]), np.max(pos[1])
        ymin, ymax = np.min(pos[0]), np.max(pos[0])
        boxes = torch.as_tensor([[xmin, ymin, xmax, ymax]], dtype=torch.float32)
        
        # 3. Get Label (ClassId)
        labels = torch.as_tensor([row['ClassId']], dtype=torch.int64)
        
        # 4. Convert Mask to torch
        masks = torch.as_tensor(mask, dtype=torch.uint8).unsqueeze(0)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks
        }

        # Convert image to tensor [0, 1]
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        return image, target

# Important: Mask R-CNN needs a custom collate_fn for the DataLoader
def collate_fn(batch):
    return tuple(zip(*batch))

In [5]:
df = pd.read_csv("data/kaggle_data/train.csv")
df.columns = ["ImageId", "ClassId", "EncodedPixels"]

steel_data = SteelDataset(df,"data/kaggle_data/train_images/")
train_loader = DataLoader(steel_data, batch_size=4, shuffle=True, collate_fn=collate_fn)
maskRcnn_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

optimizer = optim.AdamW(maskRcnn_model.parameters(), lr=1e-4, weight_decay=1e-2)
loss_fn = smp.losses.DiceLoss(mode='multiclass')

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to C:\Users\40757/.cache\torch\hub\checkpoints\maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:28<00:00, 6.32MB/s] 


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
def train_fn(epochs, train_loader, net, optimizer):
    net.to(device)
    for epoch in range(epochs):
        net.train()
        epoch_loss = 0
        
        for images, targets in train_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = net(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            epoch_loss += losses.item()

        print(f"Epoch {epoch} | Avg Loss: {epoch_loss/len(train_loader):.4f}")